## Single Responsibility
> A class should do one thing and do it well. 

If a class is handling multiple concerns, each of those concerns becomes a reason to change, making the class fragile, harder to test, and harder to maintain. As an example, consider a `Book` class:

In [ ]:
public class Book {

    private String name;
    private String author;
    private String text;

    //constructor, getters and setters
}

We can have methods which are directly related to the properties. However introducing methods not directly related to property is violation of the Single Responsibility principle:

In [ ]:
public class Book {
    //...

    void saveBookToDatabase() {
        // save current book to database
    }

    void sendPublishNotification() {
        // logic to send notifications
    }
}

The solution is to separate functionalities into a different classes:

In [ ]:
public class BookRepository {
    public void saveBook(Book book) {
        // Implementation
    }
}

public class BookNotificationService {
    void sendPublishNotification(Book book) {
        // logic to send notifications
    }
}

This separation means:
- unit testing becomes easier with clearly defined scope
- changing the class becomes easier since different functionalities are not inter-tangled
- the class is easier to understand

## Open to Extension, Closed for Modification
> "A class should be open for extension, but closed for modification."

The idea is to maintain class stability. It is better to extend a given class than to add functinality to the existing class. For example, consider the class below:

In [ ]:
public class Payment {
    public void payWithCard() {
        // Implementation
    }
    
    public void payWithPaypal() {
        // Implementation
    }
}

Now if we want to add another payment method such as cash or COD, we maybe tempted to add extra methods to the existing Payment class. However this is violation of above principle

In [ ]:
public class Payment {
    public void payWithCard(Float amount) {
        // Implementation
    }
    
    public void payWithPaypal(Float amount) {
        // Implementation
    }
    
    public void payWithCash(Float amount) {
        // Implementation
    }
    
    public void payOnDelivery(Float amount) {
        // Implementation
    }
}

The solution is to employ factory method.

In [ ]:
public interface Payment{
    public void pay(Float amount);
}

public CardPayment implements Payment{
    public void pay(Float amount){
        // Implementation
    }
}

// ... other payment classes

Another example:

In [ ]:
public class DiscountService {
    public double calculateDiscount(Customer customer) {
        if (customer.getType().equals("REGULAR")) {
            return 0.05;
        } else if (customer.getType().equals("PREMIUM")) {
            return 0.15;
        } else if (customer.getType().equals("VIP")) {
            return 0.25;
        }
        
        return 0.0;
    }
}

As new type of customers are introduced, we would need to change the logic of `calculateDiscount` method. This risks breaking implementation of existing customer types. The right way is to therefore introduce an interface:

In [ ]:
public interface DiscountPolicy {
    double getDiscount();
}

public class RegularDiscount implements DiscountPolicy {
    @Override
    public double getDiscount() { return 0.05; }
}

public class PremiumDiscount implements DiscountPolicy {
    @Override
    public double getDiscount() { return 0.15; }
}

public class VipDiscount implements DiscountPolicy {
    @Override
    public double getDiscount() { return 0.25; }
}

public class DiscountService {
    public double calculateDiscount(DiscountPolicy policy) {
        return policy.getDiscount();  // never needs to change
    }
}

New discount policies can be easily added without changing the `DiscountService` class. The advantages are:
- no change to original class
- no growing if-else chain
- new extensions can be developed independently by different teams without merge conflict

## Liskov's Substitution Principle
> "Subtypes must be substitutable for their base types without altering the correctness of the program."

If class *A* is a subtype of class *B*, then we should be able to replace *B* with *A* without disrupting the behavior of our program. Suppose we have a `Duck` class defined as:

In [ ]:
public abstract class Duck {
    public abstract void quack();
    
    public abstract void fly();

}

public class RubberDuck extends Duck {
    public void quack() {
        Person p = new Person();
        if(p.canSqueeze(this)) {
            System.out.println("Quack!");
        } else {
            throw new RuntimeException("Can't quack on its own");
        }
    }
    
    public void fly() {
        throw new RuntimeException("Can't fly");
    }

}

The Rubber duck class violates the Liskov substitution principle because a rubber duck doesn't respond correctly to the `fly` method defined by the `Duck` class. In essense, Rubber duck is not really a duck. The solution is to code by contract rather than extending. In the below code the abstractions are defined in a better manner.

In [ ]:
public interface Quackable {
    public void quack();
}

public interface Flyable {
    public void fly();
}

public class RubberDuck implements Quackable {
    public void quack() {
        Person p = new Person();
        if(p.canSqueeze(this)) {
            System.out.println("Quack!");
        } else {
            throw new RuntimeException("Can't quack on its own");
        }
    }
}

Some signs of violation of Liskov's Substitution Principle:
- overridden methods that do nothing
- overridden methods that throw `UnsupportedOperationException ` exception
- parent class method promises non-null return value, child class method returns null value

If subtypes can't be trusted to honour the parent's contract, polymorphism breaks down. The Java Collections framework violates this principle in many places:

In [ ]:
public interface Collection {
    boolean add(E e);
}

List<Integer> list = Arrays.asList(1,2,3);
list.add(5); // UnsupportedOperationException
list.remove(0); // UnsupportedOperationException

// Similar issue with Collections.unmodifiableList() or List.of()

Instead of having a `List` type, Java should have had:

In [ ]:
public interface ReadableList<E> {
    E get(int index);
    int size();
    Iterator<E> iterator();
    // ... all read operations
}

public interface MutableList<E> extends ReadableList<E> {
    boolean add(E element);
    E remove(int index);
    E set(int index, E element);
    // ... all write operations
}

## Interface Segregation
> "A class should not be forced to implement interfaces it does not use."

We should never force the client to depend on methods it doesn’t use. In short, break big interfaces into smaller logical ones. So instead of one big interface given below:

In [ ]:
public interface Operations<T> {
    public T add(T a, T b);
    
    public T subtract(T a, T b);
    
    public T multiply(T a, T b);
    
    public T divide(T a, T b);
}

Divide it into smaller interfaces, such that a class which wants to implement those functionalities can pick and choose.

In [ ]:
public interface Add<T> {
    public T add(T a, T b);
}

public interface Subtract<T> {
    public T subtract(T a, T b);
}

public interface Multiply<T> {
    public T multiply(T a, T b);
}
    
public interface Divide<T> {
    public T divide(T a, T b);
}

Violation of Interface Segregation often leads to violation of Liskov's Substitution:

In [ ]:
// If we had used the original Operations method
class KindergartenCalculator implements Operation<Integer> {
    public Integer add(Integer a, Integer b) {
        return a + b;
    }
    
    public Integer subtract(Integer a, Integer b) {
        return a - b;
    }
    
    public Integer multiply(Integer a, Integer b) {
        throw new UnsupportedOperationException("Haven't learned multiplication yet");
    }
    
    public Integer divide(Integer a, Integer b) {
        throw new UnsupportedOperationException("Haven't learned division yet");
    }
}

## Dependency Inversion
> "High-level modules should not depend on low-level modules. Both should depend on abstractions. Abstractions should not depend on details. Details should depend on abstractions."

It is a general design guideline which recommends that classes should only have direct relationships with high-level abstractions. As an example, consider the below code which violates this principle:

In [ ]:
class DatabaseUserRepository {
    public User findUserById(String username) {
        // implementation
    }
}

class UserService {
    private DatabaseUserRepository userRepository;
    
    public User authenticateUser(String username, String password) {
        User user = userRepository.findUserById(username);
        
        // ...
    }
}

`UserService` depends upon a concrete implementation. This makes it difficult to swap database implementation with another implementation. Unit testing also becomes difficult.

In [ ]:
abstract class UserRepository {
    public abstract User findUserById(String username);
}

class DatabaseUserRepository extends UserRepository {
    public User findUserById(String username) {
        // implementation
    }
}

class UserService {
    // UserService now depends upon an abstraction
    private UserRepository userRepository;
    
    public User authenticateUser(String username, String password) {
        User user = userRepository.findUserById(username);
        
        // ...
    }
}

Dependency Inversion has benefits:
- easy to do unit tests, replace the dependency with a mock
- flexibility to swap implementation without changing business logic
- loose coupling between high level module and low level module
- enforces "Open to Extension, Closed to Modification" principle, we can add new implementations without changing the code

DIP is the capstone that ties everything together:
- SRP gives you small focused classes worth depending on
- OCP lets you extend via new implementations without modification
- LSP ensures those implementations are safely substitutable
- ISP keeps the abstractions narrow so dependencies stay surgical
- DIP ensures high-level logic depends on those abstractions, so all the above benefits are actually realised at runtime